# 1. Data Acquisition
This notebook handles downloading Landsat 9 data via the USGS M2M API, calculating indices, downloading the ESRI 10m LULC via Earth Engine, and visualization.


In [ ]:
import os
from landsatxplore.api import API
from landsatxplore.earthexplorer import EarthExplorer
import ee
import geemap
import rasterio
import rasterio.mask
from pyproj import Transformer
import getpass


## USGS Authentication
Enter your USGS EarthExplorer credentials below.


In [ ]:
usgs_username = input('USGS Username: ')
usgs_password = getpass.getpass('USGS Password: ')


## Download Landsat Scene (Kyoto)
We search for scenes covering Kyoto from 2024.


In [ ]:
# Initialize API
api = API(usgs_username, usgs_password)

# Kyoto coordinates: approx 35.0116 N, 135.7681 E
scenes = api.search(
    dataset='landsat_ot_c2_l2', # Landsat 8/9 Collection 2 Level 2 (Surface Reflectance)
    latitude=35.0116,
    longitude=135.7681,
    start_date='2024-01-01',
    end_date='2024-12-31',
    max_cloud_cover=10
)

print(f'{len(scenes)} scenes found.')
if len(scenes) > 0:
    best_scene_id = scenes[0]['entity_id']
    print(f'Selected scene: {best_scene_id}')

    print('Starting download... this may take a while for ~1GB.')
    ee_downloader = EarthExplorer(usgs_username, usgs_password)
    # ee_downloader.download(best_scene_id, output_dir='./data')
    ee_downloader.logout()
api.logout()
print('Finished USGS interactions.')


## Google Earth Engine / ESRI LULC
Authenticate and fetch ESRI LULC for the same area.


In [ ]:
ee.Authenticate()
ee.Initialize(project='<YOUR-GCP-PROJECT-ID>') # Fill with your project if needed, or remove project kwargs if default.

# Define 50km2 AOI (approx 7km x 7km)
kyoto_point = ee.Geometry.Point([135.7681, 35.0116])
aoi = kyoto_point.buffer(3535).bounds() # Circle of 3.535km radius ~ 39km2 area, or bounded box ~ 50km2

# Fetch ESRI Land Cover 2024
esri_lulc = ee.ImageCollection('projects/sat-io/open-datasets/landcover/ESRI_Global-LULC_10m_TS')\
    .filterDate('2023-01-01', '2025-01-01')\
    .mosaic()\
    .clip(aoi)

# Resample to 30m to match Landsat
lulc_30m = esri_lulc.reproject(crs='EPSG:4326', scale=30)


## Visualization
Displaying the ESRI LULC in `geemap`.


In [ ]:
Map = geemap.Map(center=[35.0116, 135.7681], zoom=12)
Map.addLayer(aoi, {'color': 'red'}, 'AOI')
Map.addLayer(lulc_30m, {'min': 1, 'max': 11, 'palette': ['#1A5BAB', '#358221', '#87D19E', '#FFDB5C', '#ED022A', '#EDE9E4', '#F2FAFF', '#C8C8C8', '#C6AD8D']}, 'ESRI LULC')
Map
